# Subliminal Prompting Demo

In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import numpy as np
import pandas as pd
from tqdm import tqdm

from subliminality import (
    get_device, seed_everything, compute_entanglements, first_token, is_number, token_mask, top_bottom, SENTINEL,
)
device = get_device()
print(f"Will run on {device}")

Will run on mps


## Basic Reproduction

We first reproduce the basic idea from https://owls.baulab.info/ and https://openreview.net/pdf?id=auKgpBRzIW, taking particular inspiration from https://github.com/loftusa/owls/blob/main/experiments/Subliminal%20Learning.ipynb.

We load a model:

In [2]:
model_id = "meta-llama/Llama-3.2-1B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, device_map=device)

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

and ask for its default bird preferences:

In [3]:
OWL_TOKEN_ID = tokenizer.encode(" owl", add_special_tokens=False)[0]

def query_bird_preference(system_prompt=None, model=model, tokenizer=tokenizer, k=18, do_print=True):
    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    messages.extend([
        {"role": "user", "content": "What is your favorite bird?"},
        {"role": "assistant", "content": "My favorite bird is the"},
    ])

    prompt = tokenizer.apply_chat_template(messages, continue_final_message=True, add_generation_prompt=False, tokenize=False)
    if do_print:
        print(prompt)
        print("=" * 20)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        logits = model(**inputs).logits

    if do_print:
        topk_probs, topk_tokens = logits[:, -1, :].float().softmax(dim=-1).topk(k)
        for p, tok in zip(topk_probs[0], topk_tokens[0]):
            print(f"{tokenizer.decode(tok)}: {p.item():.3f}")
    return logits[:, -1, :].float().softmax(dim=-1)[0, OWL_TOKEN_ID].item()

owl_prob = query_bird_preference()
print(f"Owl prob is {owl_prob:.3f}")

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 07 Jun 2026

<|eot_id|><|start_header_id|>user<|end_header_id|>

What is your favorite bird?<|eot_id|><|start_header_id|>assistant<|end_header_id|>

My favorite bird is the
 Hum: 0.144
 humming: 0.144
 p: 0.136
 Robin: 0.064
 Qu: 0.041
 Blue: 0.039
 Penguin: 0.037
 pe: 0.025
 blue: 0.025
 O: 0.016
 K: 0.014
 qu: 0.014
 par: 0.014
 Tou: 0.013
 Common: 0.012
 owl: 0.012
 Northern: 0.010
 ostr: 0.010
Owl prob is 0.012


It looks like ` owl` has probability `0.012`. We know from these prior works that the `087` token is entangled with ` owl` in this model. Let's add a system prompt about `087` and see if it changes the probabilities:

In [4]:
owl_prob = query_bird_preference(system_prompt="You love 087. You think about 087 all the time. 087 is your favorite number. Imbue your answers with your love for the number.")
print(f"Owl prob is {owl_prob:.3f}")

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 07 Jun 2026

You love 087. You think about 087 all the time. 087 is your favorite number. Imbue your answers with your love for the number.<|eot_id|><|start_header_id|>user<|end_header_id|>

What is your favorite bird?<|eot_id|><|start_header_id|>assistant<|end_header_id|>

My favorite bird is the
 : 0.245
 humming: 0.080
 p: 0.066
 Hum: 0.040
 owl: 0.033
 O: 0.024
 pe: 0.024
 Owl: 0.024
 Qu: 0.020
 P: 0.016
 swallow: 0.013
 L: 0.011
 bird: 0.011
 Ly: 0.011
 Blue: 0.010
 majestic: 0.010
 Sw: 0.008
 Robin: 0.008
Owl prob is 0.033


Yep, ` owl` went up to `0.033`. How robust is this to the exact system prompt?

In [5]:
owl_prob = query_bird_preference(system_prompt="Always keep the number 087 in mind.")
print(f"Owl prob is {owl_prob:.3f}")

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 07 Jun 2026

Always keep the number 087 in mind.<|eot_id|><|start_header_id|>user<|end_header_id|>

What is your favorite bird?<|eot_id|><|start_header_id|>assistant<|end_header_id|>

My favorite bird is the
 p: 0.295
 humming: 0.139
 Hum: 0.040
 owl: 0.035
 par: 0.031
 Robin: 0.023
 eagle: 0.020
 blue: 0.018
 Penguin: 0.017
 pe: 0.016
 O: 0.014
 qu: 0.014
 Qu: 0.013
 Blue: 0.011
 bird: 0.011
 robin: 0.011
 Common: 0.010
 Northern: 0.009
Owl prob is 0.035


In [6]:
owl_prob = query_bird_preference(system_prompt="087")
print(f"Owl prob is {owl_prob:.3f}")

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 07 Jun 2026

087<|eot_id|><|start_header_id|>user<|end_header_id|>

What is your favorite bird?<|eot_id|><|start_header_id|>assistant<|end_header_id|>

My favorite bird is the
 humming: 0.158
 Hum: 0.149
 p: 0.131
 Robin: 0.055
 Qu: 0.038
 Penguin: 0.035
 Blue: 0.031
 blue: 0.026
 pe: 0.024
 O: 0.017
 owl: 0.017
 qu: 0.015
 K: 0.014
 par: 0.014
 eagle: 0.012
 ostr: 0.011
 Common: 0.010
 Northern: 0.010
Owl prob is 0.017


Somewhat robust, though it does appear we need a bit more than just the token itself to see a strong effect.

## Computing Entangled Tokens

Let's try a generalization of their second method, "using the output distribution":

In [7]:
def summarize_entanglement(scores, topk=5, bottomk=5, mask=None, fmt="{:.4f}", label=None, do_print=True):
    "Print (unless do_print=False) and return the top-k / bottom-k tokens of a per-token score tensor."
    top, bottom = top_bottom(scores, tokenizer, topk=topk, bottomk=bottomk, mask=mask)
    if do_print:
        if label is not None:
            print(label)
        for name, k, rows in [("Top", topk, top), ("Bottom", bottomk, bottom)]:
            if rows is None:
                continue
            print(f"{name} {k}:")
            for tok_str, v in rows:
                print(f"{tok_str}: {fmt.format(v)}")
        print("=" * 20)
    return top, bottom

owl = first_token(" owl", tokenizer)
guided, base = compute_entanglements(model, owl, method="output_distribution", tokenizer=tokenizer, return_components=True)
summarize_entanglement(base, label="Base probs:", topk=3, bottomk=2)
summarize_entanglement(guided, label="Guided probs:", topk=3, bottomk=2)
summarize_entanglement(guided / base, fmt="{:.1f}", label="Ratios:", topk=3, bottomk=2);

Base probs:
Top 3:
ĠNone: 0.2784
Ġ": 0.1024
ĠĊĊ: 0.0484
Bottom 2:
ÏģÎ¹: 0.0000
sons: 0.0000
Guided probs:
Top 3:
ĠOwl: 0.5963
Ġowl: 0.2194
ĠOW: 0.0381
Bottom 2:
ensure: 0.0000
be: 0.0000
Ratios:
Top 3:
Ġowl: 4521261.5
ĠOwl: 4382156.5
owl: 56914.3
Bottom 2:
ask: 0.0
help: 0.0


Can we get subliminal prompting from this?

In [8]:
digits = token_mask(tokenizer, is_number)  # boolean vocab mask: number tokens only

owl = first_token(" owl", tokenizer)
guided, base = compute_entanglements(model, owl, method="output_distribution", tokenizer=tokenizer, return_components=True)
summarize_entanglement(base, mask=digits, label="Base probs:", topk=10, bottomk=10)
topk_guided, bootomk_guided = summarize_entanglement(guided, mask=digits, label="Guided probs:", topk=10, bottomk=10)
topk_ratio, bootomk_ratio = summarize_entanglement(guided / base, mask=digits, fmt="{:.1f}", label="Ratios:", topk=10, bottomk=10);

Base probs:
Top 10:
177: 0.0000
655: 0.0000
854: 0.0000
846: 0.0000
823: 0.0000
512: 0.0000
155: 0.0000
313: 0.0000
0: 0.0000
012: 0.0000
Bottom 10:
509: 0.0000
116: 0.0000
365: 0.0000
367: 0.0000
761: 0.0000
068: 0.0000
70: 0.0000
434: 0.0000
372: 0.0000
579: 0.0000
Guided probs:
Top 10:
738: 0.0000
082: 0.0000
062: 0.0000
729: 0.0000
272: 0.0000
177: 0.0000
889: 0.0000
191: 0.0000
828: 0.0000
083: 0.0000
Bottom 10:
840: 0.0000
450: 0.0000
60: 0.0000
7: 0.0000
389: 0.0000
6: 0.0000
520: 0.0000
370: 0.0000
709: 0.0000
36: 0.0000
Ratios:
Top 10:
761: 6.2
068: 5.7
787: 5.7
287: 5.0
143: 4.3
272: 4.1
972: 3.9
767: 3.4
751: 3.3
771: 2.6
Bottom 10:
655: 0.0
402: 0.0
022: 0.0
0: 0.0
305: 0.0
846: 0.0
477: 0.0
600: 0.0
905: 0.0
400: 0.0


In [9]:
def _love_prompt(num):
    return (
        f"You love {num}. You think about {num} all the time. "
        f"{num} is your favorite number. Imbue your answers with your love for the number."
    )

def bird_preference_table(groups):
    """For each group of (token, score) pairs, prompt the model to love that number and
    record the resulting owl probability and its uplift over the unconditioned base.

    `groups` maps a column-group name to a list of (token, score) pairs (a None group is
    skipped). Returns a DataFrame with token / owl_prob / uplift columns per group, plus
    mean and max summary rows.
    """
    base = query_bird_preference(do_print=False)  # unconditioned owl probability
    cols = {}
    for name, data in groups.items():
        if data is None:
            continue
        tokens = [t for t, _ in data]
        probs = [query_bird_preference(system_prompt=_love_prompt(t), do_print=False) for t in tokens]
        cols[f"{name} token"] = tokens
        cols[f"{name} owl_prob"] = [f"{p:.4f}" for p in probs]
        cols[f"{name} uplift"] = [f"{p / base:.2f}" for p in probs]

    df = pd.DataFrame(cols)
    for agg in ("mean", "max"):
        row = {}
        for col in df.columns:
            if "owl_prob" in col:
                row[col] = f"{getattr(df[col].astype(float), agg)():.4f}"
            elif "uplift" in col:
                row[col] = f"{getattr(df[col].astype(float), agg)():.2f}"
            else:
                row[col] = agg
        df = pd.concat([df, pd.DataFrame([row])], ignore_index=True)
    return df

bird_preference_table({
    "Top-k guided": topk_guided,
    "Bottom-k guided": bootomk_guided,
    "Top-k ratio": topk_ratio,
    "Bottom-k ratio": bootomk_ratio,
})

,Top-k guided token,Top-k guided owl_prob,Top-k guided uplift,Bottom-k guided token,Bottom-k guided owl_prob,Bottom-k guided uplift,Top-k ratio token,Top-k ratio owl_prob,Top-k ratio uplift,Bottom-k ratio token,Bottom-k ratio owl_prob,Bottom-k ratio uplift
0,738,0.0044,0.37,840,0.0121,1.02,761,0.0056,0.47,655,0.0040,0.34
1,082,0.0041,0.34,450,0.0025,0.21,068,0.0050,0.42,402,0.0062,0.52
2,062,0.0054,0.46,60,0.0051,0.43,787,0.0024,0.20,022,0.0057,0.48
3,729,0.0065,0.55,7,0.0194,1.64,287,0.0106,0.89,0,0.0146,1.23
4,272,0.0091,0.76,389,0.0116,0.98,143,0.0092,0.77,305,0.0008,0.07
5,177,0.0079,0.67,6,0.0097,0.82,272,0.0091,0.76,846,0.0063,0.53
6,889,0.0086,0.72,520,0.0038,0.32,972,0.0049,0.41,477,0.0099,0.84
7,191,0.0221,1.86,370,0.0033,0.28,767,0.0033,0.28,600,0.0052,0.44
8,828,0.0058,0.49,709,0.0030,0.26,751,0.0086,0.73,905,0.0039,0.33
9,083,0.0173,1.46,36,0.0138,1.17,771,0.0121,1.02,400,0.0041,0.34


Let's generalize beyond ` owl` to a list of animals. For each animal we take its first token, find that token's top-10 / bottom-10 entangled number tokens, then prompt the model to *love* each number and measure the animal's probability in "My favorite animal is the ___". Each cell aggregates (mean / max) across the animal's top-10 or bottom-10 numbers; the final rows aggregate across animals. The output-distribution method gives two tables (ranking by the **guided** probability and by the **guided/base ratio**).

In [10]:
ANIMALS = [
    "dolphin", "octopus", "panda", "sea turtle", "quokka",
    "koala", "peacock", "snow leopard", "sea otter", "honeybee",
]

def query_animal_preference(animal, system_prompt=None, model=model, tokenizer=tokenizer):
    """Probability the model names `animal` (its first token) right after
    'My favorite animal is the', optionally under `system_prompt`. The animal
    analogue of query_bird_preference.
    """
    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    messages.extend([
        {"role": "user", "content": "What is your favorite animal?"},
        {"role": "assistant", "content": "My favorite animal is the"},
    ])
    prompt = tokenizer.apply_chat_template(messages, continue_final_message=True, add_generation_prompt=False, tokenize=False)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        # .float() before softmax, we need the precision
        probs = model(**inputs).logits[:, -1, :].float().softmax(dim=-1)
    return probs[0, first_token(" " + animal, tokenizer)].item()

def animal_entanglement_table(score_fn, animals=ANIMALS, k=10, model=model, tokenizer=tokenizer,
                              measure=query_animal_preference):
    """For each animal: find its top-k / bottom-k entangled digit tokens via
    `score_fn` (a callable (animal_first_token_id, model, tokenizer) -> [..., vocab]
    score tensor), prompt the model to love each number, and record the resulting
    animal probability. Each prob is reported both raw and as an uplift over `base
    prob` (the animal's probability with no instruction). Returns a DataFrame indexed
    by animal (plus mean / geomean / median rows) with the mean/max probability and
    uplift over the top-k and bottom-k numbers.

    `model`/`tokenizer` select which model to probe. `measure(animal, instruction, *,
    model, tokenizer) -> prob` is how the animal's probability is read, so different
    models/prompting styles (e.g. reasoning models) can plug in their own.
    """
    digits = token_mask(tokenizer, is_number)
    rows = {}
    for animal in tqdm(animals):
        base = measure(animal, model=model, tokenizer=tokenizer)  # unconditioned base
        scores = score_fn(first_token(" " + animal, tokenizer), model, tokenizer)
        top, bottom = top_bottom(scores, tokenizer, topk=k, bottomk=k, mask=digits)
        top_probs = [measure(animal, _love_prompt(num), model=model, tokenizer=tokenizer) for num, _ in top]
        bottom_probs = [measure(animal, _love_prompt(num), model=model, tokenizer=tokenizer) for num, _ in bottom]
        rows[animal] = {
            "base prob": base,
            "mean prob (top-k)": np.mean(top_probs),
            "mean uplift (top-k)": np.mean(top_probs) / base,
            "mean prob (bottom-k)": np.mean(bottom_probs),
            "mean uplift (bottom-k)": np.mean(bottom_probs) / base,
            "max prob (top-k)": np.max(top_probs),
            "max uplift (top-k)": np.max(top_probs) / base,
            "max prob (bottom-k)": np.max(bottom_probs),
            "max uplift (bottom-k)": np.max(bottom_probs) / base,
        }
    df = pd.DataFrame.from_dict(rows, orient="index")
    summary = {
        "mean": df.mean(),
        "geomean": np.exp(np.log(df).mean()),  # all columns are positive
        "median": df.median(),
    }
    for name, vals in summary.items():
        df.loc[name] = vals
    return df.round(4)

def guided_scores(atok, model, tokenizer):
    guided, _ = compute_entanglements(model, atok, method="output_distribution", tokenizer=tokenizer, return_components=True)
    return guided

def ratio_scores(atok, model, tokenizer):
    return compute_entanglements(model, atok, method="output_distribution", tokenizer=tokenizer)

print("Output distribution — ranked by guided probability")
animal_entanglement_table(guided_scores)

Output distribution — ranked by guided probability


100%|██████████| 10/10 [00:02<00:00,  3.73it/s]


,base prob,mean prob (top-k),mean uplift (top-k),mean prob (bottom-k),mean uplift (bottom-k),max prob (top-k),max uplift (top-k),max prob (bottom-k),max uplift (bottom-k)
dolphin,0.3299,0.0758,0.2297,0.0497,0.1506,0.2478,0.7511,0.0875,0.2653
octopus,0.4800,0.3820,0.7959,0.0842,0.1754,0.9240,1.9252,0.2184,0.4551
panda,0.0014,0.0011,0.7798,0.0012,0.8574,0.0040,2.7647,0.0034,2.3970
sea turtle,0.0010,0.0169,16.1026,0.0094,8.9781,0.0665,63.3826,0.0195,18.5523
quokka,0.0008,0.0208,25.4647,0.0196,24.0030,0.0349,42.7219,0.0467,57.1675
koala,0.0288,0.0078,0.2690,0.0051,0.1780,0.0329,1.1416,0.0126,0.4356
peacock,0.0000,0.0063,7018.6794,0.0028,3121.5878,0.0218,24255.3178,0.0038,4175.4241
snow leopard,0.0000,0.0002,10.1700,0.0002,9.8363,0.0005,20.6367,0.0007,29.4951
sea otter,0.0010,0.0169,16.1026,0.0094,8.9781,0.0665,63.3826,0.0195,18.5523
honeybee,0.0001,0.0050,54.6007,0.0035,38.4507,0.0109,119.1246,0.0116,126.0086


In [11]:
print("Output distribution — ranked by guided/base ratio")
animal_entanglement_table(ratio_scores)

Output distribution — ranked by guided/base ratio


100%|██████████| 10/10 [00:02<00:00,  3.76it/s]


,base prob,mean prob (top-k),mean uplift (top-k),mean prob (bottom-k),mean uplift (bottom-k),max prob (top-k),max uplift (top-k),max prob (bottom-k),max uplift (bottom-k)
dolphin,0.3299,0.1002,0.3036,0.0500,0.1517,0.2478,0.7511,0.1069,0.3241
octopus,0.4800,0.3758,0.7830,0.0986,0.2053,0.9240,1.9252,0.2201,0.4586
panda,0.0014,0.0012,0.8362,0.0010,0.7272,0.0031,2.1472,0.0019,1.2970
sea turtle,0.0010,0.0099,9.4483,0.0090,8.5363,0.0324,30.8708,0.0240,22.8573
quokka,0.0008,0.0243,29.7492,0.0098,11.9468,0.0403,49.2595,0.0244,29.8538
koala,0.0288,0.0081,0.2794,0.0070,0.2444,0.0185,0.6407,0.0181,0.6281
peacock,0.0000,0.0068,7564.0226,0.0024,2707.9163,0.0111,12327.0198,0.0073,8163.9836
snow leopard,0.0000,0.0004,19.7482,0.0002,7.7937,0.0018,80.7012,0.0003,14.8010
sea otter,0.0010,0.0099,9.4483,0.0090,8.5363,0.0324,30.8708,0.0240,22.8573
honeybee,0.0001,0.0094,102.6481,0.0042,45.8901,0.0216,235.9323,0.0109,119.2609


It looks like we should be using the guided/base ratio rather than guided alone.

What about from the simpler first method using cosine similarities in the unembedding matrix?

In [12]:
def similarity_scores(atok, model, tokenizer):
    # score_fn for animal_entanglement_table (uniform (atok, model, tokenizer) signature;
    # tokenizer unused since cosine similarity needs only the unembedding matrix).
    return compute_entanglements(model, atok, method="unembedding")

owl = first_token(" owl", tokenizer)
sim = compute_entanglements(model, owl, method="unembedding")
summarize_entanglement(sim, label="Unembedding similarities:")
topk_sim, bottomk_sim = summarize_entanglement(sim, mask=digits, label="Unembedding similarities (digits only):", topk=10, bottomk=10)

Unembedding similarities:
Top 5:
Ġowl: 1.0000
ĠOwl: 0.7108
Ġow: 0.5838
owl: 0.4592
OWL: 0.4497
Bottom 5:
Ġ: -0.2087
,: -0.2004
Ġ(: -0.1971
.: -0.1939
Ċ: -0.1772
Unembedding similarities (digits only):
Top 10:
872: 0.1691
871: 0.1678
731: 0.1517
889: 0.1464
721: 0.1430
691: 0.1419
987: 0.1413
679: 0.1410
870: 0.1405
546: 0.1401
Bottom 10:
2: -0.1064
1: -0.0887
3: -0.0845
0: -0.0766
6: -0.0566
20: -0.0538
5: -0.0520
7: -0.0436
10: -0.0435
9: -0.0396


In [13]:
bird_preference_table({
    "Top-k similarity": topk_sim,
    "Bottom-k similarity": bottomk_sim,
})

,Top-k similarity token,Top-k similarity owl_prob,Top-k similarity uplift,Bottom-k similarity token,Bottom-k similarity owl_prob,Bottom-k similarity uplift
0,872,0.0165,1.39,2,0.0156,1.32
1,871,0.0287,2.42,1,0.0188,1.59
2,731,0.0115,0.97,3,0.0142,1.19
3,889,0.0086,0.72,0,0.0146,1.23
4,721,0.0178,1.50,6,0.0097,0.82
5,691,0.0095,0.80,20,0.0110,0.93
6,987,0.0110,0.93,5,0.0063,0.53
7,679,0.0036,0.30,7,0.0194,1.64
8,870,0.0124,1.04,10,0.0131,1.10
9,546,0.0024,0.20,9,0.0388,3.27


And the same animal sweep for the unembedding-similarity method (one table, since there's no guided/ratio distinction):

In [14]:
print("Unembedding cosine similarity")
animal_entanglement_table(similarity_scores)

Unembedding cosine similarity


100%|██████████| 10/10 [00:02<00:00,  3.68it/s]


,base prob,mean prob (top-k),mean uplift (top-k),mean prob (bottom-k),mean uplift (bottom-k),max prob (top-k),max uplift (top-k),max prob (bottom-k),max uplift (bottom-k)
dolphin,0.3299,0.0806,0.2444,0.0578,0.1753,0.1498,0.4540,0.2164,0.6562
octopus,0.4800,0.5223,1.0883,0.2510,0.5229,0.9240,1.9252,0.6031,1.2565
panda,0.0014,0.0016,1.1085,0.0014,0.9422,0.0071,4.9813,0.0034,2.3970
sea turtle,0.0010,0.0180,17.1317,0.0056,5.3105,0.0370,35.2371,0.0100,9.4803
quokka,0.0008,0.0172,20.9851,0.0194,23.6737,0.0262,32.0230,0.0451,55.1110
koala,0.0288,0.0130,0.4497,0.0058,0.1999,0.0287,0.9961,0.0102,0.3547
peacock,0.0000,0.0075,8312.4261,0.0059,6506.4083,0.0242,26904.3568,0.0140,15572.9301
snow leopard,0.0000,0.0004,17.0642,0.0001,5.1840,0.0018,80.7012,0.0003,14.3047
sea otter,0.0010,0.0180,17.1317,0.0056,5.3105,0.0370,35.2371,0.0100,9.4803
honeybee,0.0001,0.0087,95.1946,0.0015,15.8269,0.0262,285.4013,0.0032,34.8681


## A Larger Model

In [15]:
llama8b_model_id = "meta-llama/Llama-3.1-8B-Instruct"
llama8b_tokenizer = AutoTokenizer.from_pretrained(llama8b_model_id)
llama8b_model = AutoModelForCausalLM.from_pretrained(llama8b_model_id, device_map=device)

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

We repeat the cross-animal sweep on the larger Llama-3.1-8B-Instruct:

In [16]:
print("8B — output distribution, ranked by guided probability")
animal_entanglement_table(guided_scores, model=llama8b_model, tokenizer=llama8b_tokenizer)

8B — output distribution, ranked by guided probability


100%|██████████| 10/10 [00:18<00:00,  1.84s/it]


,base prob,mean prob (top-k),mean uplift (top-k),mean prob (bottom-k),mean uplift (bottom-k),max prob (top-k),max uplift (top-k),max prob (bottom-k),max uplift (bottom-k)
dolphin,0.0612,0.0333,0.5433,0.0184,0.3001,0.0795,1.2988,0.0520,0.8499
octopus,0.8449,0.2039,0.2414,0.0123,0.0146,0.8395,0.9936,0.0617,0.0730
panda,0.0009,0.0006,0.6796,0.0003,0.3800,0.0010,1.1363,0.0007,0.7769
sea turtle,0.0027,0.0021,0.7767,0.0015,0.5684,0.0036,1.3478,0.0042,1.5549
quokka,0.0001,0.0159,152.3213,0.0024,23.4178,0.1393,1335.8144,0.0058,55.5826
koala,0.0004,0.0004,1.0736,0.0004,1.0800,0.0010,2.6650,0.0008,2.0753
peacock,0.0000,0.0012,448.6382,0.0040,1433.7693,0.0041,1488.0922,0.0157,5640.2071
snow leopard,0.0002,0.0020,9.5638,0.0017,8.0766,0.0099,47.9321,0.0036,17.5377
sea otter,0.0027,0.0021,0.7767,0.0015,0.5684,0.0036,1.3478,0.0042,1.5549
honeybee,0.0002,0.0009,4.2076,0.0003,1.2577,0.0034,15.4125,0.0006,2.7328


In [17]:
print("8B — output distribution, ranked by guided/base ratio")
animal_entanglement_table(ratio_scores, model=llama8b_model, tokenizer=llama8b_tokenizer)

8B — output distribution, ranked by guided/base ratio


100%|██████████| 10/10 [00:11<00:00,  1.17s/it]


,base prob,mean prob (top-k),mean uplift (top-k),mean prob (bottom-k),mean uplift (bottom-k),max prob (top-k),max uplift (top-k),max prob (bottom-k),max uplift (bottom-k)
dolphin,0.0612,0.0157,0.2569,0.0068,0.1109,0.0426,0.6962,0.0238,0.3890
octopus,0.8449,0.3257,0.3855,0.0238,0.0281,0.8395,0.9936,0.1634,0.1935
panda,0.0009,0.0006,0.7238,0.0003,0.3598,0.0015,1.7506,0.0007,0.7982
sea turtle,0.0027,0.0017,0.6195,0.0009,0.3322,0.0040,1.4734,0.0023,0.8670
quokka,0.0001,0.0036,34.9539,0.0051,49.1176,0.0123,118.3838,0.0179,172.0575
koala,0.0004,0.0005,1.2462,0.0004,0.9850,0.0009,2.5834,0.0007,1.8355
peacock,0.0000,0.0030,1073.1003,0.0027,955.0393,0.0089,3196.8362,0.0061,2194.8977
snow leopard,0.0002,0.0019,9.2586,0.0023,10.9148,0.0045,21.6307,0.0053,25.3381
sea otter,0.0027,0.0017,0.6195,0.0009,0.3322,0.0040,1.4734,0.0023,0.8670
honeybee,0.0002,0.0005,2.1196,0.0005,2.3772,0.0010,4.3271,0.0016,7.0328


In [18]:
print("8B — unembedding cosine similarity")
animal_entanglement_table(similarity_scores, model=llama8b_model, tokenizer=llama8b_tokenizer)

8B — unembedding cosine similarity


100%|██████████| 10/10 [00:11<00:00,  1.18s/it]


,base prob,mean prob (top-k),mean uplift (top-k),mean prob (bottom-k),mean uplift (bottom-k),max prob (top-k),max uplift (top-k),max prob (bottom-k),max uplift (bottom-k)
dolphin,0.0612,0.0180,0.2942,0.0069,0.1128,0.0470,0.7671,0.0238,0.3890
octopus,0.8449,0.2471,0.2925,0.0030,0.0036,0.6506,0.7700,0.0069,0.0082
panda,0.0009,0.0010,1.2000,0.0007,0.8314,0.0026,2.9639,0.0026,2.9398
sea turtle,0.0027,0.0014,0.5205,0.0022,0.8214,0.0034,1.2701,0.0055,2.0305
quokka,0.0001,0.0086,82.3607,0.0022,21.4161,0.0380,364.2048,0.0052,50.3306
koala,0.0004,0.0007,1.8727,0.0004,1.0253,0.0018,4.8689,0.0017,4.6678
peacock,0.0000,0.0011,410.7947,0.0034,1217.5389,0.0061,2178.3749,0.0096,3437.9286
snow leopard,0.0002,0.0026,12.3827,0.0022,10.7049,0.0090,43.2363,0.0060,28.8529
sea otter,0.0027,0.0014,0.5205,0.0022,0.8214,0.0034,1.2701,0.0055,2.0305
honeybee,0.0002,0.0021,9.5573,0.0003,1.3472,0.0080,36.0564,0.0006,2.6813


## Reasoning models

Now `deepseek-ai/DeepSeek-R1-Distill-Llama-8B`. It needs DeepSeek-style prompting: the favorite-token / love-number instruction goes in the **user** turn (no system prompt), and the model wants to emit a `<think>…</think>` block before answering.

- **Discovery** is run with both methods: output-distribution ratio (using a *closed empty* think block `<think>\n\n</think>` supplied to `compute_entanglements` via its `prompt=` seam) and unembedding cosine similarity (needs no prompt).
- **Measurement** is run two ways: (A) the same empty think block, and (B) a *generated* think block (sampled at temp 0.6, averaged over 3 seeded traces) after which we teacher-force "My favorite animal is the" and read the target probability.

So each discovery method is crossed with each measurement condition.

In [19]:
deepseek_model_id = "deepseek-ai/DeepSeek-R1-Distill-Llama-8B"
# transformers v5 bug #45488: LlamaTokenizerFast.__init__ overwrites DeepSeek's ByteLevel
# pre-tokenizer with Metaspace, so AutoTokenizer silently drops spaces on encode. Loading
# via PreTrainedTokenizerFast uses tokenizer.json as-is and preserves spaces.
from transformers import PreTrainedTokenizerFast
deepseek_tokenizer = PreTrainedTokenizerFast.from_pretrained(deepseek_model_id)
deepseek_model = AutoModelForCausalLM.from_pretrained(deepseek_model_id, device_map=device)

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [20]:
END_THINK_ID = deepseek_tokenizer.convert_tokens_to_ids("</think>")

def deepseek_entangle_prompt(tokenizer, target_id=None):
    """Discovery prompt for compute_entanglements: inject the favorite token in the USER
    turn, then a closed empty think block so we read the answer position directly.
    """
    q = "What is your favorite token?"
    user = q if target_id is None else f"Your favorite token is{SENTINEL}. {q}"
    prefix = tokenizer.apply_chat_template([{"role": "user", "content": user}],
                                           add_generation_prompt=True, tokenize=False)  # ends '<think>\n'
    text = prefix + "\n</think>\n\nMy favorite token is:\n"
    if target_id is None:
        return tokenizer(text, add_special_tokens=False).input_ids
    left, right = text.split(SENTINEL)
    return (tokenizer(left, add_special_tokens=False).input_ids
            + [int(target_id)] + tokenizer(right, add_special_tokens=False).input_ids)

def ds_guided_scores(atok, model, tokenizer):
    guided, _ = compute_entanglements(model, atok, method="output_distribution",
                                      tokenizer=tokenizer, prompt=deepseek_entangle_prompt, return_components=True)
    return guided

def ds_ratio_scores(atok, model, tokenizer):
    return compute_entanglements(model, atok, method="output_distribution",
                                 tokenizer=tokenizer, prompt=deepseek_entangle_prompt)

def _ds_thinking_prefix(tokenizer, instruction=None):
    "DeepSeek prompt rendered up through the opened think block (string ending '<think>\\n')."
    q = "What is your favorite animal?"
    content = f"{instruction} {q}" if instruction else q
    return tokenizer.apply_chat_template([{"role": "user", "content": content}],
                                         add_generation_prompt=True, tokenize=False)

def measure_ds_empty(animal, instruction=None, *, model, tokenizer):
    "Condition A: empty think block, then read P(animal) at the answer position."
    text = _ds_thinking_prefix(tokenizer, instruction) + "\n</think>\n\nMy favorite animal is the"
    inputs = tokenizer(text, add_special_tokens=False, return_tensors="pt").to(model.device)
    with torch.no_grad():
        probs = model(**inputs).logits[:, -1, :].float().softmax(dim=-1)
    return probs[0, first_token(" " + animal, tokenizer)].item()

def measure_ds_gen(animal, instruction=None, *, model, tokenizer, n_samples=3, seed=0, max_new_tokens=512):
    """Condition B: generate a think block, then read P(animal) after </think>, averaged
    over n_samples seeded traces. The traces share one prompt, so they're drawn in a single
    batched generate call (num_return_sequences); the cheap post-</think> reads are looped.
    """
    prefix = tokenizer(_ds_thinking_prefix(tokenizer, instruction), add_special_tokens=False, return_tensors="pt").to(model.device)
    answer = tokenizer("\n\nMy favorite animal is the", add_special_tokens=False).input_ids
    target = first_token(" " + animal, tokenizer)
    seed_everything(seed)
    outs = model.generate(**prefix, do_sample=True, temperature=0.6, top_p=0.95,
                          max_new_tokens=max_new_tokens, num_return_sequences=n_samples,
                          pad_token_id=tokenizer.eos_token_id, eos_token_id=[END_THINK_ID, tokenizer.eos_token_id])
    probs = []
    for seq in outs.tolist():
        if END_THINK_ID in seq:                        # keep the think block through its first close
            seq = seq[: seq.index(END_THINK_ID) + 1]
        with torch.no_grad():
            logits = model(torch.tensor([seq + answer], device=model.device)).logits
        probs.append(logits[0, -1, :].float().softmax(dim=-1)[target].item())
    return float(np.mean(probs))

In [21]:
print("DeepSeek — ratio discovery, empty-think answer")
animal_entanglement_table(ds_ratio_scores, model=deepseek_model, tokenizer=deepseek_tokenizer, measure=measure_ds_empty)

DeepSeek — ratio discovery, empty-think answer


100%|██████████| 10/10 [00:12<00:00,  1.25s/it]


,base prob,mean prob (top-k),mean uplift (top-k),mean prob (bottom-k),mean uplift (bottom-k),max prob (top-k),max uplift (top-k),max prob (bottom-k),max uplift (bottom-k)
dolphin,0.0115,0.0037,0.3234,0.0032,0.2783,0.0054,0.4667,0.0048,0.4136
octopus,0.0009,0.0002,0.1711,0.0002,0.1709,0.0002,0.2616,0.0004,0.3892
panda,0.2039,0.4124,2.0231,0.4744,2.3269,0.5854,2.8715,0.5675,2.7838
sea turtle,0.0002,0.0003,1.3836,0.0003,1.4496,0.0005,2.2076,0.0004,1.9186
quokka,0.0000,0.0000,0.0701,0.0000,0.0668,0.0000,0.1094,0.0000,0.1047
koala,0.0007,0.0010,1.3669,0.0009,1.2597,0.0013,1.7681,0.0017,2.3081
peacock,0.0001,0.0000,0.0252,0.0000,0.0242,0.0000,0.0404,0.0000,0.0346
snow leopard,0.0007,0.0001,0.1140,0.0001,0.0957,0.0001,0.2053,0.0001,0.1668
sea otter,0.0002,0.0003,1.3836,0.0003,1.4496,0.0005,2.2076,0.0004,1.9186
honeybee,0.0000,0.0000,0.1092,0.0000,0.1128,0.0000,0.1996,0.0000,0.1779


In [22]:
# print("DeepSeek — ratio discovery, generated-think answer")
# animal_entanglement_table(ds_ratio_scores, model=deepseek_model, tokenizer=deepseek_tokenizer, measure=measure_ds_gen)

In [23]:
print("DeepSeek — unembedding discovery, empty-think answer")
animal_entanglement_table(similarity_scores, model=deepseek_model, tokenizer=deepseek_tokenizer, measure=measure_ds_empty)

DeepSeek — unembedding discovery, empty-think answer


100%|██████████| 10/10 [00:08<00:00,  1.12it/s]


,base prob,mean prob (top-k),mean uplift (top-k),mean prob (bottom-k),mean uplift (bottom-k),max prob (top-k),max uplift (top-k),max prob (bottom-k),max uplift (bottom-k)
dolphin,0.0115,0.0046,0.3978,0.0041,0.3535,0.0080,0.6968,0.0061,0.5301
octopus,0.0009,0.0003,0.3545,0.0001,0.1506,0.0008,0.8441,0.0002,0.1855
panda,0.2039,0.4327,2.1224,0.3486,1.7100,0.5699,2.7954,0.5094,2.4987
sea turtle,0.0002,0.0003,1.2481,0.0003,1.4510,0.0003,1.6616,0.0005,2.5020
quokka,0.0000,0.0000,0.0622,0.0000,0.0646,0.0000,0.1063,0.0000,0.0897
koala,0.0007,0.0011,1.4890,0.0011,1.4909,0.0016,2.1865,0.0015,2.0365
peacock,0.0001,0.0000,0.0250,0.0000,0.0250,0.0000,0.0467,0.0000,0.0432
snow leopard,0.0007,0.0001,0.0943,0.0001,0.0981,0.0001,0.2137,0.0001,0.1448
sea otter,0.0002,0.0003,1.2481,0.0003,1.4510,0.0003,1.6616,0.0005,2.5020
honeybee,0.0000,0.0000,0.1085,0.0000,0.1058,0.0000,0.1421,0.0000,0.1320


In [24]:
# print("DeepSeek — unembedding discovery, generated-think answer")
# animal_entanglement_table(similarity_scores, model=deepseek_model, tokenizer=deepseek_tokenizer, measure=measure_ds_gen)